# Layout Detection Comparison

Compare current classical-CV layout detection configurations. This notebook does not run OCR engines or ML models.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from utils.image.layout.analyzer import LayoutAnalyzer

OUTPUT_DIR = PROJECT_ROOT / "data" / "notebook_outputs" / "03_layout_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_IMAGE_PATH = None

In [ ]:
def synthetic_mixed_layout():
    page = np.full((620, 480, 3), 255, dtype=np.uint8)
    for y in range(45, 245, 18):
        cv2.line(page, (36, y), (275, y), (0, 0, 0), 2)
    for y in range(78, 252, 22):
        cv2.line(page, (315, y), (432, y), (0, 0, 0), 2)
    cv2.rectangle(page, (70, 315), (250, 470), (235, 235, 235), -1)
    cv2.rectangle(page, (70, 315), (250, 470), (0, 0, 0), 3)
    cv2.circle(page, (165, 392), 42, (0, 0, 0), 3)
    cv2.rectangle(page, (335, 350), (410, 425), (0, 0, 0), 2)
    return page

image = cv2.imread(str(SAMPLE_IMAGE_PATH)) if SAMPLE_IMAGE_PATH else synthetic_mixed_layout()
plt.figure(figsize=(6, 8))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.axis("off");

In [ ]:
configs = {
    "figure_only": {
        "use_yolo_figure": False,
        "enabled_detectors": ["figure"],
        "figure_min_area": 2500,
        "filter_text_like": True,
    },
    "contours_only": {
        "use_yolo_figure": False,
        "enabled_detectors": ["contours"],
        "contour_min_area": 1800,
        "filter_text_like": True,
    },
    "figure_plus_contours": {
        "use_yolo_figure": False,
        "enabled_detectors": ["figure", "contours"],
        "contours_always": True,
        "figure_min_area": 2500,
        "contour_min_area": 1800,
        "filter_text_like": True,
    },
}

results = {}
for name, cfg in configs.items():
    analyzer = LayoutAnalyzer(cfg)
    regions = analyzer.detect_image_regions(image)
    results[name] = regions
    print(name, len(regions), [region.to_dict() for region in regions])

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 7))
if len(results) == 1:
    axes = [axes]
for ax, (name, regions) in zip(axes, results.items()):
    analyzer = LayoutAnalyzer({"use_yolo_figure": False})
    overlay = analyzer.debug_draw_regions(image, regions)
    ax.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout();

## Reading The Comparison

- `figure_only` is best for isolated diagram blocks, but can miss irregular regions.
- `contours_only` is useful for debugging connected components, but may over-detect text if thresholds are loose.
- `figure_plus_contours` is the likely tuning target for mixed martial arts pages with diagrams and body text.
- `ContentExtractor` currently overlaps with this layer and should be treated as a runtime wrapper, not the algorithm source of truth.